# 3-stage: локальная визуализация предсказания

Положи в папку `3-stage/` три файла:

```text
best_model.pth
image.npy
mask.npy
```

Ноутбук загружает модель, выполняет inference и показывает одно изображение:

- CT-срез — серый фон;
- настоящая маска — синяя область;
- предсказанная маска — красная область.

В заголовке выводятся `Dice`, `Precision` и `Recall`.


## 1. Импорты и пути


In [ ]:
from pathlib import Path
import importlib.util
import matplotlib.pyplot as plt
from matplotlib.patches import Patch
import numpy as np
import torch


def find_repo_root() -> Path:
    cwd = Path.cwd().resolve()
    for candidate in [cwd, *cwd.parents]:
        if (candidate / "2-stage" / "model.py").exists() and (candidate / "3-stage").exists():
            return candidate
    raise FileNotFoundError(
        "Не найден 2-stage/model.py. Запусти notebook из репозитория Lung-Tumor-Segmentation."
    )


REPO_ROOT = find_repo_root()
STAGE_DIR = REPO_ROOT / "3-stage"
CHECKPOINT_PATH = STAGE_DIR / "best_model.pth"
IMAGE_PATH = STAGE_DIR / "image.npy"
MASK_PATH = STAGE_DIR / "mask.npy"

for path, label in [
    (CHECKPOINT_PATH, "model"),
    (IMAGE_PATH, "image"),
    (MASK_PATH, "mask"),
]:
    if not path.exists():
        raise FileNotFoundError(f"Не найден {label}: {path}")

MODEL_FILE = REPO_ROOT / "2-stage" / "model.py"
spec = importlib.util.spec_from_file_location("lung_unet_model", MODEL_FILE)
if spec is None or spec.loader is None:
    raise ImportError(f"Не удалось загрузить model.py: {MODEL_FILE}")
model_module = importlib.util.module_from_spec(spec)
spec.loader.exec_module(model_module)
build_model = model_module.build_model

print("model:", CHECKPOINT_PATH)
print("image:", IMAGE_PATH)
print("mask:", MASK_PATH)


## 2. Вспомогательные функции


In [ ]:
def load_checkpoint(path: Path):
    try:
        return torch.load(path, map_location="cpu", weights_only=False)
    except TypeError:
        return torch.load(path, map_location="cpu")


def choose_device():
    if torch.cuda.is_available():
        return torch.device("cuda:0")
    if hasattr(torch.backends, "mps") and torch.backends.mps.is_available():
        return torch.device("mps")
    return torch.device("cpu")


def as_chw_float32(array: np.ndarray) -> np.ndarray:
    array = np.asarray(array)
    if array.ndim == 2:
        array = array[None, :, :]
    elif array.ndim == 3 and array.shape[0] == 1:
        pass
    elif array.ndim == 3 and array.shape[-1] == 1:
        array = np.moveaxis(array, -1, 0)
    else:
        raise ValueError(f"Ожидался массив [H, W] или [1, H, W], получено: {array.shape}")
    return array.astype(np.float32, copy=False)


def masked(mask: np.ndarray):
    return np.ma.masked_where(mask == 0, mask)


def compute_metrics(pred_mask: np.ndarray, gt_mask: np.ndarray):
    pred = pred_mask.astype(bool)
    gt = gt_mask.astype(bool)
    tp = float(np.logical_and(pred, gt).sum())
    fp = float(np.logical_and(pred, ~gt).sum())
    fn = float(np.logical_and(~pred, gt).sum())
    eps = 1e-7
    return {
        "dice": (2.0 * tp) / (2.0 * tp + fp + fn + eps),
        "precision": tp / (tp + fp + eps),
        "recall": tp / (tp + fn + eps),
    }


## 3. Загрузка модели и inference


In [ ]:
checkpoint = load_checkpoint(CHECKPOINT_PATH)
config = checkpoint.get("config", {})
img_size = int(config.get("data", {}).get("img_size", 512))
threshold = float(checkpoint.get("best_threshold", config.get("metrics", {}).get("threshold", 0.5)))
device = choose_device()

model = build_model(config.get("model", {})).to(device)
model.load_state_dict(checkpoint["model_state_dict"])
model.eval()

image_chw = as_chw_float32(np.load(IMAGE_PATH, allow_pickle=False))
gt_mask = (as_chw_float32(np.load(MASK_PATH, allow_pickle=False))[0] > 0.5).astype(np.uint8)
expected_shape = (1, img_size, img_size)
if tuple(image_chw.shape) != expected_shape:
    raise ValueError(f"Image shape {image_chw.shape}, expected {expected_shape}")
if tuple(gt_mask.shape) != expected_shape[1:]:
    raise ValueError(f"Mask shape {gt_mask.shape}, expected {expected_shape[1:]}")

x = torch.from_numpy(np.ascontiguousarray(image_chw)).unsqueeze(0).to(device)
with torch.no_grad():
    logits = model(x)
    probability = torch.sigmoid(logits)[0, 0].detach().cpu().numpy().astype(np.float32)

pred_mask = (probability >= threshold).astype(np.uint8)
metrics = compute_metrics(pred_mask, gt_mask)

print("device:", device)
print("checkpoint epoch:", checkpoint.get("epoch"))
print("best val dice:", checkpoint.get("best_val_dice"))
print("threshold:", threshold)
print("Dice:", metrics["dice"])
print("Precision:", metrics["precision"])
print("Recall:", metrics["recall"])


## 4. Overlay-визуализация

Синяя область — настоящая маска. Красная область — предсказанная маска.


In [ ]:
fig, ax = plt.subplots(figsize=(9, 9))
ax.imshow(image_chw[0], cmap="gray", vmin=0, vmax=1)
ax.imshow(masked(gt_mask), cmap="Blues", alpha=0.55, vmin=0, vmax=1)
ax.imshow(masked(pred_mask), cmap="Reds", alpha=0.55, vmin=0, vmax=1)
ax.set_title(
    f"Dice={metrics['dice']:.3f} | "
    f"Precision={metrics['precision']:.3f} | "
    f"Recall={metrics['recall']:.3f} | "
    f"threshold={threshold:.3f}"
)
ax.legend(
    handles=[
        Patch(facecolor="tab:blue", alpha=0.55, label="Ground truth"),
        Patch(facecolor="tab:red", alpha=0.55, label="Prediction"),
    ],
    loc="lower right",
)
ax.axis("off")
fig.tight_layout()
plt.show()
